## Handling the Extract and Transform part

In [92]:
import os
import numpy as np
import pandas as pd

import chardet
import seaborn as sns

In [27]:
def detect_encoding(file):
    detector = chardet.universaldetector.UniversalDetector()
    with open(file, "rb") as f:
        for line in f:
            detector.feed(line)
            if detector.done:
                break
    detector.close()
    return detector.result

In [32]:
loc = "C:\Offline_Docs\Personal\DT_CS_2023\DE_CS_202309\Project\Case_Study_202309_Data"
# loc = r"F:\github\DE_CS_202309\Project\Case_Study_202309_Data"

enc_list = {}

In [30]:
# Detect Character Encoding

file = loc+'\\'+ os.listdir(loc)[0]
encoding = detect_encoding(file)['encoding']
# print("Encoding:", encoding['encoding'])
encoding

'ascii'

In [33]:
count = 0
filenames = []

for file in os.listdir(loc):
  if file.endswith(".csv"):
    filenames.append(file)
    count+=1
    encoding = detect_encoding(loc+'\\'+file)['encoding']
    enc_list[file]=encoding

print(f'Total {count} CSV files present')

Total 130 CSV files present


In [62]:
df_enc_lst= pd.DataFrame.from_dict(enc_list,orient='Index')
df_enc_lst.reset_index(inplace=True)
df_enc_lst.columns=['File_Name','Encoding']
df_enc_lst.head()

,File_Name,Encoding
0,201901_Orders_2019_02_01_03_10_55.csv,ascii
1,201901_Orders_2019_02_04_15_41_32.csv,ascii
2,201902_Orders_2019_03_04_02_26_31.csv,Windows-1252
3,201903_Orders_2019_04_01_20_39_17.csv,Windows-1252
4,201903_Orders_2019_04_04_08_51_54.csv,ISO-8859-1
...,...,...
125,202211_Orders_2022_12_02_21_54_39.csv,Windows-1252
126,202211_Orders_2022_12_05_21_17_28.csv,ISO-8859-1
127,202211_Orders_2022_12_08_15_01_57.csv,Windows-1252
128,202211_Orders_2022_12_11_08_20_07.csv,Windows-1252


In [63]:
df_enc_lst['Encoding'].value_counts()

Windows-1252    68
ISO-8859-1      53
ascii            9
Name: Encoding, dtype: int64

In [41]:
df1 = pd.read_csv(loc+'\\'+ os.listdir(loc)[0], delimiter='|', encoding='utf8')
df1.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0,0,0.2,5.5512
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0,0,0.2,4.2717
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0,0,0.2,-64.7748
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0,0,0.8,-5.4870
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0,0,0.2,4.8840


In [42]:
mega_df = pd.DataFrame(columns=df1.columns)
mega_df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [104]:
type(np.count_nonzero(filenames))

int

In [108]:
missing_cols = []

for row, file in enumerate(filenames):
  tmp_df = pd.read_csv(loc+'\\'+ file, delimiter='|',encoding=enc_list[file])
  # pd.concat([mega_df, tmp_df])
  cnt_cols = np.count_nonzero(tmp_df.columns)
  if cnt_cols != 21:
    print(f'Total Columns, {np.count_nonzero(tmp_df.columns)}, Filename - {file}')
    missing_cols.append(file)
    continue
  else:
    if row != np.count_nonzero(filenames)-1:
      mega_df = pd.concat([mega_df,tmp_df])


Total Columns, 20, Filename - 202206_Orders_2022_07_05_18_29_34.csv


In [83]:
missing_cols

['202206_Orders_2022_07_05_18_29_34.csv']

In [106]:
mega_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 87371 entries, 0 to 446
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order ID       87371 non-null  object 
 1   Order Date     87371 non-null  object 
 2   Ship Date      87371 non-null  object 
 3   Ship Mode      87371 non-null  object 
 4   Customer ID    87371 non-null  object 
 5   Customer Name  87371 non-null  object 
 6   Segment        87371 non-null  object 
 7   Country        87371 non-null  object 
 8   City           87371 non-null  object 
 9   State          87371 non-null  object 
 10  Postal Code    87371 non-null  object 
 11  Region         87371 non-null  object 
 12  Product ID     87371 non-null  object 
 13  Category       87371 non-null  object 
 14  Sub-Category   87371 non-null  object 
 15  Product Name   87371 non-null  object 
 16  Sales          87371 non-null  float64
 17  Quantity       87371 non-null  float64
 18  Discount

In [86]:
mega_df.drop(columns=['Row ID'], inplace=True)

In [90]:
mega_df[['Sales','Quantity']] = mega_df[['Sales','Quantity']].astype(float)

In [91]:
mega_df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0.0,0.0,0.2,5.5512
1,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0.0,0.0,0.2,4.2717
2,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0.0,0.0,0.2,-64.7748
3,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0.0,0.0,0.8,-5.4870
4,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0.0,0.0,0.2,4.8840


## Adding Data to a Database